# Flow Map Models Tutorial: CM, CTM & MeanFlow on 2D Two Moons

**Chapter 11 — Learning Fast Generators from Scratch**

### Model quick reference

**CM — §11.2**

Network:
$$
\mathbf{f}_\theta(\mathbf{x}_s, s)
$$
Approximate target:
$$
\mathbf{x}_0
$$

**CTM (v-prediction) — §11.4**

Network:
$$
\mathbf{G}_\theta(\mathbf{x}_s, s, t)
$$
Approximate target:
$$
\Psi_{s \to t}(\mathbf{x}_s)
$$

**MF — §11.5**

Network:
$$
\mathbf{h}_\theta(\mathbf{x}_s, s, t)
$$
Approximate target: average drift
$$
\mathbf{h}^*
$$

**Shared forward process** (flow-matching schedule):

$$\mathbf{x}_s = \alpha_s\,\mathbf{x}_0 + \sigma_s\,\boldsymbol{\epsilon}, \qquad \alpha_s = 1-s,\;\; \sigma_s = s, \qquad s \in [0, 1], \qquad \boldsymbol{\epsilon} \sim \mathcal{N}(\mathbf{0}, \mathbf{I})$$

- $s = 0$: clean data $\mathbf{x}_0$
- $s = T = 1$: pure noise $\boldsymbol{\epsilon}$

> **Colab rendering note.** Math-heavy comparison tables are written as stacked formula blocks instead of Markdown/HTML tables. Colab often wraps MathJax inside table cells, which can split formulas in awkward places.

## 0. Setup & Imports

In [ ]:
import math
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons

# ---- Reproducibility ----
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

## 1. Visualization Helpers

In [ ]:
def plot_samples(data_np, samples_dict, title='', figsize_per=4.5):
    n = len(samples_dict)
    fig, axes = plt.subplots(1, n, figsize=(figsize_per * n, 4))
    if n == 1: axes = [axes]
    for ax, (label, pts) in zip(axes, samples_dict.items()):
        ax.scatter(data_np[:, 0], data_np[:, 1], s=1, alpha=0.12, c='gray', label='data')
        ax.scatter(pts[:, 0], pts[:, 1], s=2, alpha=0.45, label=label)
        ax.set_title(label, fontsize=10)
        ax.set_xlim(-5, 7); ax.set_ylim(-4, 4); ax.set_aspect('equal')
        ax.legend(fontsize=7, loc='upper left')
    if title: fig.suptitle(title, fontsize=12, fontweight='bold', y=1.02)
    fig.tight_layout(); plt.show()

def plot_losses(loss_dict):
    fig, ax = plt.subplots(figsize=(7, 3.5))
    for label, losses in loss_dict.items():
        ax.plot(losses, label=label, alpha=0.8, linewidth=0.7)
    ax.set_xlabel('step'); ax.set_ylabel('loss'); ax.set_yscale('log')
    ax.legend(); ax.set_title('Training Losses'); fig.tight_layout(); plt.show()

## 2. Dataset & Forward Process

**Forward corruption:**
$$\mathbf{x}_s = (1 - s)\,\mathbf{x}_0 + s\,\boldsymbol{\epsilon}, \qquad \boldsymbol{\epsilon} \sim \mathcal{N}(\mathbf{0}, \mathbf{I})$$

**Conditional velocity** (constant along the straight interpolation path):
$$\mathbf{v}_{\text{cond}} = \alpha_s'\,\mathbf{x}_0 + \sigma_s'\,\boldsymbol{\epsilon} = -\mathbf{x}_0 + \boldsymbol{\epsilon} = \boldsymbol{\epsilon} - \mathbf{x}_0$$

In [ ]:
def make_dataset(n_samples=100_000, noise=0.05):
    """Generate two-moons with modes well separated."""
    X, _ = make_moons(n_samples=n_samples, noise=noise, random_state=SEED)
    X = X * 2; X[:, 1] -= 0.5
    return torch.tensor(X, dtype=torch.float32)

def forward_process(x_0, eps, s):
    """x_s = (1-s)*x_0 + s*eps"""
    return (1.0 - s) * x_0 + s * eps

dataset = make_dataset(100_000)
fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(dataset[:5000, 0], dataset[:5000, 1], s=1, alpha=0.3)
ax.set_title('Two Moons (scaled)'); ax.set_aspect('equal')
plt.tight_layout(); plt.show()

## 3. Shared Network Architecture

All three models share the same MLP backbone. Each scalar time condition is embedded via **sinusoidal positional encoding** and concatenated with the spatial input $\mathbf{x}$ before the hidden layers.

In [ ]:
class SinusoidalTimeEmbedding(nn.Module):
    def __init__(self, dim=64):
        super().__init__()
        self.dim = dim
    def forward(self, t):
        half = self.dim // 2
        freqs = torch.exp(-math.log(10_000) * torch.arange(half, device=t.device, dtype=t.dtype) / half)
        args = t.unsqueeze(-1) * freqs
        return torch.cat([args.sin(), args.cos()], dim=-1)

class MLP(nn.Module):
    def __init__(self, data_dim=2, n_time_conds=1, hidden_dim=256, n_layers=4, time_emb_dim=64):
        super().__init__()
        self.time_embs = nn.ModuleList([SinusoidalTimeEmbedding(time_emb_dim) for _ in range(n_time_conds)])
        in_dim = data_dim + time_emb_dim * n_time_conds
        layers = []
        for i in range(n_layers):
            layers.append(nn.Linear(in_dim if i == 0 else hidden_dim, hidden_dim))
            layers.append(nn.SiLU())
        layers.append(nn.Linear(hidden_dim, data_dim))
        self.net = nn.Sequential(*layers)
    def forward(self, x, *times):
        embs = [fn(t) for fn, t in zip(self.time_embs, times)]
        return self.net(torch.cat([x] + embs, dim=-1))

## 4. Consistency Model (CM) — §11.2

### Goal

$$\mathbf{f}_\theta(\mathbf{x}_s, s) \;\approx\; \Psi_{s \to 0}(\mathbf{x}_s) \;=\; \mathbf{x}_0$$

### Parameterisation (guarantees boundary $\mathbf{f}(\mathbf{x}_0, 0) = \mathbf{x}_0$)

$$\mathbf{f}_\theta(\mathbf{x}, s) = (1 - s)\,\mathbf{x} + s\,F_\theta(\mathbf{x}, s)$$

### CT Loss (Eq. 11.2.6)

Sample the **same** pair $(\mathbf{x}_0, \boldsymbol{\epsilon})$; construct $\mathbf{x}_s$ and $\mathbf{x}_{s'}$ on the same path at adjacent noise levels $s > s'$:

$$\mathcal{L}_{\text{CT}}^N(\boldsymbol{\theta}, \boldsymbol{\theta}^-) = \mathbb{E}_{\mathbf{x}_0, \boldsymbol{\epsilon}, i}\!\left[\omega(s_i)\;d\!\left(\mathbf{f}_\theta(\mathbf{x}_{s_{i+1}}, s_{i+1}),\;\mathbf{f}_{\theta^-}(\mathbf{x}_{s_i}, s_i)\right)\right]$$

### ECT Tricks (Consistency Models Made Easy)

1. **Sigmoid schedule** (ECT Eq. 15): $s'/s$ starts at $0$ (denoising) $\to$ $\approx\!1$ (tight consistency)
2. **Adaptive weighting** (ECT Eq. 16): $w(\Delta) = 1 / \sqrt{\|\Delta\|^2 + c^2}$
3. **LogNormal time sampling** for $p(s)$

### Sampling (Algorithm 9)

- **1-step**: $\hat{\mathbf{x}}_0 = \mathbf{f}_\theta(\hat{\mathbf{x}}_T, T)$
- **Multi-step**: $\gamma = 1$ **only** — CM cannot do $\gamma < 1$

In [ ]:
class ConsistencyModel(nn.Module):
    """CM: f_theta(x_s, s) = Psi_{s->0}(x_s) = x_0  (Section 11.2)
    Training with ECT tricks: sigmoid schedule, adaptive weighting, LogNormal."""
    def __init__(self, data_dim=2):
        super().__init__()
        self.net = MLP(data_dim=data_dim, n_time_conds=1)

    def f_theta(self, x_s, s):
        s_col = s.unsqueeze(-1)
        F = self.net(x_s, s)
        return (1.0 - s_col) * x_s + s_col * F

    def training_loss(self, x_0, step, total_steps):
        B = x_0.shape[0]; dev = x_0.device
        eps = torch.randn_like(x_0)
        s_min = 1e-3

        # LogNormal time sampling
        log_s = -1.0 + 1.5 * torch.randn(B, device=dev)
        s = log_s.exp().clamp(s_min, 1.0)

        # ECT sigmoid mapping schedule (Eq. 15)
        k = 8.0
        progress = step / max(total_steps - 1, 1)
        n_val = 1.0 + k / (1.0 + math.exp(-12.0 * (progress - 0.5)))
        q = 2.0
        ratio = max(0.0, min(1.0 - 1.0 / (q ** n_val), 1.0 - 1e-4))
        s_prime = (s * ratio).clamp(min=0.0)

        x_s  = forward_process(x_0, eps, s.unsqueeze(-1))
        x_sp = forward_process(x_0, eps, s_prime.unsqueeze(-1))

        pred = self.f_theta(x_s, s)
        with torch.no_grad():
            target = self.f_theta(x_sp, s_prime)

        delta = pred - target
        c = 0.1
        adaptive_w = 1.0 / (delta.detach().pow(2).sum(-1, keepdim=True) + c**2).sqrt()
        time_w = 1.0 / (s.unsqueeze(-1) * n_val + 1e-8)
        return (adaptive_w * time_w * delta.pow(2)).mean()

## 5. Consistency Trajectory Model (CTM, v-prediction) — §11.4

### Goal

$$\mathbf{G}_\theta(\mathbf{x}_s, s, t) \;\approx\; \Psi_{s \to t}(\mathbf{x}_s) \quad \text{for any } t \le s$$

### v-prediction parameterisation

$$\boxed{\mathbf{G}_\theta(\mathbf{x}_s, s, t) = \mathbf{x}_s + (t - s)\,\mathbf{v}_\theta(\mathbf{x}_s, s, t)}$$

- **Boundary**: $\mathbf{G}_\theta(\mathbf{x}_s, s, s) = \mathbf{x}_s$ ✓
- **Optimal**: $\mathbf{v}_\theta^* = \mathbb{E}[\boldsymbol{\epsilon} - \mathbf{x}_0 \mid \mathbf{x}_s]$

### Self-induced velocity (Eq. 11.4.6)

At $t = s$, the v-prediction naturally recovers the instantaneous velocity:

$$\mathbf{v}_\theta(\mathbf{x}_s, s, s) \;\approx\; \mathbf{v}^*(\mathbf{x}_s, s)$$

This defines a **self-induced ODE**: $\frac{\mathrm{d}\mathbf{x}}{\mathrm{d}\tau} = \mathbf{v}_{\theta^-}(\mathbf{x}, \tau, \tau)$

### Training: semigroup loss via self-induced solver (Eq. 11.4.7)

**Key difference from CM**: the intermediate state $\mathbf{x}_u$ is obtained by **running the self-induced ODE solver** from $\mathbf{x}_s$, NOT by interpolating the forward process with the same $(\mathbf{x}_0, \boldsymbol{\epsilon})$.

One Euler step from $s$ to $u$ ($t \le u \le s$):

$$\mathbf{x}_u = \mathbf{x}_s + (u - s)\,\mathbf{v}_{\theta^-}(\mathbf{x}_s, s, s)$$

**Semigroup loss**: the direct jump $s \to t$ should match two hops $s \to u \to t$:

$$\mathcal{L}_{\text{semi}} = \mathbb{E}\!\left[\omega(s)\;d\!\left(\mathbf{G}_\theta(\mathbf{x}_s, s, t),\;\mathbf{G}_{\theta^-}(\mathbf{x}_u, u, t)\right)\right]$$

### Auxiliary diffusion loss (§11.4.3, Eq. 11.4.8)

$$\mathcal{L}_{\text{DM}}(\theta) = \mathbb{E}\!\left[w(s)\;\left\|\mathbf{v}_\theta(\mathbf{x}_s, s, s) - (\boldsymbol{\epsilon} - \mathbf{x}_0)\right\|_2^2\right]$$

**Total**: $\mathcal{L} = \mathcal{L}_{\text{semi}} + \lambda\,\mathcal{L}_{\text{DM}}$

### CM vs CTM intermediate state

**CM uses a forward-process intermediate point** on the same $(\mathbf{x}_0, \boldsymbol{\epsilon})$ path:

$$
\mathbf{x}_{s'} = (1-s')\,\mathbf{x}_0 + s'\,\boldsymbol{\epsilon}
$$

**CTM uses a self-induced solver intermediate point** produced by stepping the learned ODE from $s$ to $u$:

$$
\mathbf{x}_u = \mathbf{x}_s + (u-s)\,\mathbf{v}_{\theta^-}(\mathbf{x}_s, s, s)
$$


In [ ]:
class CTM(nn.Module):
    """CTM (v-prediction): G_theta(x_s, s, t) = Psi_{s->t}(x_s)  (Section 11.4)

    Key difference from CM: the intermediate state x_u is obtained by
    running the SELF-INDUCED ODE solver (one Euler step), NOT by
    interpolating the forward process with the same (x_0, eps).

    Training: semigroup loss + auxiliary diffusion loss (Section 11.4.3).
    """
    def __init__(self, data_dim=2, lambda_dm=1.0):
        super().__init__()
        self.net = MLP(data_dim=data_dim, n_time_conds=2)
        self.lambda_dm = lambda_dm

    def v_theta(self, x_s, s, t):
        return self.net(x_s, s, t)

    def G_theta(self, x_s, s, t):
        return x_s + (t - s).unsqueeze(-1) * self.v_theta(x_s, s, t)

    def training_loss(self, x_0, step, total_steps):
        B = x_0.shape[0]; dev = x_0.device
        eps = torch.randn_like(x_0)
        s_min = 1e-3

        # LogNormal time sampling
        log_s = -1.0 + 1.5 * torch.randn(B, device=dev)
        s = log_s.exp().clamp(s_min, 1.0)

        # ECT sigmoid schedule for u (intermediate time)
        k = 8.0
        progress = step / max(total_steps - 1, 1)
        n_val = 1.0 + k / (1.0 + math.exp(-12.0 * (progress - 0.5)))
        q = 2.0
        ratio = max(0.0, min(1.0 - 1.0 / (q ** n_val), 1.0 - 1e-4))
        u = (s * ratio).clamp(min=0.0)   # intermediate time, t <= u <= s

        t = torch.rand(B, device=dev) * u  # target time, t in [0, u]
        v_cond = eps - x_0

        x_s = forward_process(x_0, eps, s.unsqueeze(-1))

        # ---- Self-induced solver: one Euler step from s to u (Eq. 11.4.6) ----
        # The self-induced ODE velocity at time s is v_theta(x_s, s, s).
        # One Euler step:  x_u = x_s + (u - s) * v_{theta^-}(x_s, s, s)
        # Note: u < s, so (u - s) < 0 — this steps backward in noise level.
        with torch.no_grad():
            v_self = self.v_theta(x_s, s, s)                    # self-induced velocity
            x_u = x_s + (u - s).unsqueeze(-1) * v_self          # Euler step s -> u

        # ---- Semigroup loss (Eq. 11.4.7) ----
        # G_theta(x_s, s, t) should match G_{theta^-}(x_u, u, t)
        pred = self.G_theta(x_s, s, t)
        with torch.no_grad():
            target = self.G_theta(x_u, u, t)

        delta = pred - target
        c = 0.1
        adaptive_w = 1.0 / (delta.detach().pow(2).sum(-1, keepdim=True) + c**2).sqrt()
        time_w = 1.0 / (s.unsqueeze(-1) * n_val + 1e-8)
        semigroup_loss = (adaptive_w * time_w * delta.pow(2)).mean()

        # ---- Auxiliary diffusion loss (Eq. 11.4.8) ----
        # v_theta(x_s, s, s) should match v_cond = eps - x_0
        v_at_s = self.v_theta(x_s, s, s)
        dm_loss = (v_at_s - v_cond).pow(2).mean()

        return semigroup_loss + self.lambda_dm * dm_loss

## 6. MeanFlow (MF) — §11.5

### Average drift

$$\mathbf{h}^*(\mathbf{x}_s, s, t) := \frac{1}{t - s}\int_s^{t} \mathbf{v}^*(\mathbf{x}_u, u)\,\mathrm{d}u$$

$$\Psi_{s \to t}(\mathbf{x}_s) = \mathbf{x}_s + (t - s)\,\mathbf{h}^*(\mathbf{x}_s, s, t)$$

### MeanFlow Identity

$$\boxed{\mathbf{h} = \mathbf{v}^*(\mathbf{x}_s, s) - (s - t)\!\left(\mathbf{v}^*(\mathbf{x}_s, s)\,\partial_{\mathbf{x}}\mathbf{h} + \partial_s \mathbf{h}\right)}$$

### Training loss

$$\mathcal{L}_{\text{MF}}(\theta) = \mathbb{E}_{s, \mathbf{x}_s}\!\left[w(s)\;\left\|\mathbf{h}_\theta(\mathbf{x}_s, s, t) - \mathbf{h}^{\text{tgt}}_{\theta^-}\right\|_2^2\right], \quad \mathbf{h}^{\text{tgt}}_{\theta^-} = \mathbf{v}_{\text{cond}} - (s - t)\!\left(\mathbf{v}_{\text{cond}}\,\partial_{\mathbf{x}}\mathbf{h}_{\theta^-} + \partial_s \mathbf{h}_{\theta^-}\right)$$

### Flow-matching ratio (MeanFlow paper Tab. 1a)

With probability $p_{\text{fm}}$, set $t = s$. Then $(s - t) = 0$ kills the JVP term, so the loss becomes **standard flow matching**: $\|\mathbf{h}_\theta(\mathbf{x}_s, s, s) - \mathbf{v}_{\text{cond}}\|^2$.

### Flow-matching ratio guide

- $p_{\text{fm}} = 0\%$: 1-step generation **fails**.
- $p_{\text{fm}} = 25\%$: **optimal** balance between MeanFlow identity training and flow-matching anchoring.
- $p_{\text{fm}} = 100\%$: reduces to standard flow matching, which usually needs many steps.


In [ ]:
class MeanFlow(nn.Module):
    """MeanFlow: h_theta(x_s, s, t) = average drift  (Section 11.5)
    With probability p_fm, sets t=s (flow-matching mode)."""
    def __init__(self, data_dim=2, p_fm=0.25):
        super().__init__()
        self.data_dim = data_dim
        self.net = MLP(data_dim=data_dim, n_time_conds=2)
        self.p_fm = p_fm

    def h_theta(self, x_s, s, t):
        return self.net(x_s, s, t)

    def training_loss(self, x_0):
        B, D = x_0.shape; dev = x_0.device
        eps = torch.randn_like(x_0)
        s_min = 1e-3
        s = torch.rand(B, device=dev) * (1.0 - s_min) + s_min
        v_cond = eps - x_0

        # Per-sample: flow-matching (t=s) or MeanFlow (t<s)
        is_fm = torch.rand(B, device=dev) < self.p_fm
        t = torch.where(is_fm, s, torch.rand(B, device=dev) * s)

        x_s = forward_process(x_0, eps, s.unsqueeze(-1))

        # Total derivative dh/ds via autograd
        if (~is_fm).any():
            x_ad = x_s.detach().requires_grad_(True)
            s_ad = s.detach().requires_grad_(True)
            t_det = t.detach()
            h_for_grad = self.net(x_ad, s_ad, t_det)
            dhds_parts = []
            for i in range(D):
                grad_x, grad_s = torch.autograd.grad(
                    h_for_grad[:, i].sum(), [x_ad, s_ad], retain_graph=True)
                dhds_parts.append((v_cond * grad_x).sum(dim=-1) + grad_s)
            dhds = torch.stack(dhds_parts, dim=-1).detach()
        else:
            dhds = torch.zeros_like(x_s)

        h = self.net(x_s, s, t)
        d = (s - t).unsqueeze(-1)
        h_tgt = v_cond - d * dhds
        return (h - h_tgt.detach()).pow(2).mean()

## 7. Training Loop

In [ ]:
def train_model(model, model_type, dataset, total_steps=10_000,
                batch_size=2048, lr=1e-3, device='cpu'):
    model = model.to(device); dataset = dataset.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    losses = []; model.train()
    for step in range(total_steps):
        idx = torch.randint(0, len(dataset), (batch_size,), device=device)
        x_0 = dataset[idx]
        if model_type in ('cm', 'ctm'):
            loss = model.training_loss(x_0, step, total_steps)
        else:
            loss = model.training_loss(x_0)
        opt.zero_grad(); loss.backward(); opt.step()
        losses.append(loss.item())
        if (step + 1) % 2000 == 0 or step == 0:
            print(f'  [{model_type.upper():>3s}] step {step+1:>5d}/{total_steps}'
                  f'  loss = {loss.item():.4f}')
    model.eval(); return losses

## 8. Train All Three Models

~1–3 min on Colab GPU.

In [ ]:
import time

STEPS = 10_000; BS = 2048; LR = 1e-3; N = 5000
data_np = dataset.numpy()
timings = {}

print('=' * 50); print('  Training CM'); print('=' * 50)
cm = ConsistencyModel()
t0 = time.time()
cm_losses = train_model(cm, 'cm', dataset, STEPS, BS, LR, device)
timings['CM'] = time.time() - t0
print(f'  CM training time: {timings["CM"]:.1f}s')

print('\n' + '=' * 50); print('  Training CTM (+ aux diffusion loss)'); print('=' * 50)
ctm = CTM(lambda_dm=1.0)
t0 = time.time()
ctm_losses = train_model(ctm, 'ctm', dataset, STEPS, BS, LR, device)
timings['CTM'] = time.time() - t0
print(f'  CTM training time: {timings["CTM"]:.1f}s')

print('\n' + '=' * 50); print('  Training MeanFlow (25% FM ratio)'); print('=' * 50)
mf = MeanFlow(p_fm=0.25)
t0 = time.time()
mf_losses = train_model(mf, 'mf', dataset, STEPS, BS, LR, device)
timings['MF'] = time.time() - t0
print(f'  MF training time: {timings["MF"]:.1f}s')

print('\n' + '=' * 50)
for name, t in timings.items():
    print(f'    {name:>3s}: {t:.1f}s')
print('=' * 50)

plot_losses({'CM': cm_losses, 'CTM': ctm_losses, 'MF': mf_losses})

## 9. Unified Sampler with $\gamma$-Sampling

At each step $s \to \tau$ (for CTM and MF):

$$\mathbf{x}_\tau = \alpha_\tau \hat{\mathbf{x}}_0 + \sigma_\tau\!\left(\sqrt{1 - \gamma^2}\;\boldsymbol{\epsilon}_{\text{det}} + \gamma\;\boldsymbol{\epsilon}_{\text{new}}\right), \qquad \boldsymbol{\epsilon}_{\text{det}} = \frac{\hat{\mathbf{x}}_\tau^{\text{det}} - \alpha_\tau \hat{\mathbf{x}}_0}{\sigma_\tau}$$

**CM** can **ONLY** do $\gamma = 1$ for multi-step (only predicts $\hat{\mathbf{x}}_0$, no intermediate states).

**CTM & MF** support any $\gamma \in [0, 1]$: $\gamma\!=\!0$ deterministic, $\gamma\!=\!1$ stochastic.

In [ ]:
@torch.no_grad()
def sample(model, model_type, n_samples=5000, steps=1, gamma=0.0, data_dim=2):
    dev = next(model.parameters()).device
    model.eval()
    ts = torch.linspace(1.0, 0.0, steps + 1).tolist()
    x = torch.randn(n_samples, data_dim, device=dev)

    if model_type == 'cm':
        assert steps == 1 or gamma == 1.0, (
            f"CM multi-step requires gamma=1 (fully stochastic), got gamma={gamma}.")
        s_cur = torch.full((n_samples,), ts[0], device=dev)
        x_clean = model.f_theta(x, s_cur)
        if steps == 1: return x_clean
        for i in range(1, steps):
            tau = ts[i]
            if tau <= 1e-6: break
            eps_new = torch.randn_like(x_clean)
            x = (1.0 - tau) * x_clean + tau * eps_new
            x_clean = model.f_theta(x, torch.full((n_samples,), tau, device=dev))
        return x_clean

    elif model_type == 'ctm':
        for i in range(steps):
            s_val, tau_val = ts[i], ts[i + 1]
            s = torch.full((n_samples,), s_val, device=dev)
            if tau_val <= 1e-6 or gamma == 0.0:
                x = model.G_theta(x, s, torch.full((n_samples,), tau_val, device=dev))
            else:
                t_tau  = torch.full((n_samples,), tau_val, device=dev)
                t_zero = torch.zeros(n_samples, device=dev)
                x_det   = model.G_theta(x, s, t_tau)
                x_clean = model.G_theta(x, s, t_zero)
                eps_det = (x_det - (1.0 - tau_val) * x_clean) / tau_val
                eps_new = torch.randn_like(x)
                x = ((1.0 - tau_val) * x_clean
                     + tau_val * (math.sqrt(1.0 - gamma**2) * eps_det + gamma * eps_new))
        return x

    elif model_type == 'mf':
        for i in range(steps):
            s_val, tau_val = ts[i], ts[i + 1]
            s = torch.full((n_samples,), s_val, device=dev)
            t = torch.full((n_samples,), tau_val, device=dev)
            if tau_val <= 1e-6 or gamma == 0.0:
                x = x - (s_val - tau_val) * model.h_theta(x, s, t)
            else:
                t_zero  = torch.zeros(n_samples, device=dev)
                x_det   = x - (s_val - tau_val) * model.h_theta(x, s, t)
                x_clean = x - s_val * model.h_theta(x, s, t_zero)
                eps_det = (x_det - (1.0 - tau_val) * x_clean) / tau_val
                eps_new = torch.randn_like(x)
                x = ((1.0 - tau_val) * x_clean
                     + tau_val * (math.sqrt(1.0 - gamma**2) * eps_det + gamma * eps_new))
        return x

## 10. One-Step Generation

**CM**

$$
\hat{\mathbf{x}}_0 = \mathbf{f}_\theta(\hat{\mathbf{x}}_T, T)
$$

**CTM**

$$
\hat{\mathbf{x}}_0 = \mathbf{G}_\theta(\hat{\mathbf{x}}_T, T, 0)
$$

**MF**

$$
\hat{\mathbf{x}}_0 = \mathbf{x}_T - T\,\mathbf{h}_\theta(\mathbf{x}_T, T, 0)
$$


In [ ]:
cm_1  = sample(cm,  'cm',  N, steps=1).cpu().numpy()
ctm_1 = sample(ctm, 'ctm', N, steps=1, gamma=0.0).cpu().numpy()
mf_1  = sample(mf,  'mf',  N, steps=1, gamma=0.0).cpu().numpy()

plot_samples(data_np, {
    'CM (1-step)': cm_1, 'CTM (1-step)': ctm_1, 'MF (1-step)': mf_1
}, title='One-Step Generation')

## 11. Multi-Step Generation

**CM**: $\gamma = 1$ **only**.  **CTM & MF**: any $\gamma$.

In [ ]:
for n_steps in [2, 5, 10]:
    print(f'\n-- {n_steps}-step generation --')
    cm_ms = sample(cm,  'cm',  N, steps=n_steps, gamma=1.0).cpu().numpy()
    ctm_d = sample(ctm, 'ctm', N, steps=n_steps, gamma=0.0).cpu().numpy()
    ctm_s = sample(ctm, 'ctm', N, steps=n_steps, gamma=1.0).cpu().numpy()
    mf_d  = sample(mf,  'mf',  N, steps=n_steps, gamma=0.0).cpu().numpy()
    mf_s  = sample(mf,  'mf',  N, steps=n_steps, gamma=1.0).cpu().numpy()

    plot_samples(data_np, {
        f'CM {n_steps}s \u03b3=1 (only)': cm_ms,
        f'CTM {n_steps}s \u03b3=0': ctm_d,
        f'CTM {n_steps}s \u03b3=1': ctm_s,
        f'MF {n_steps}s \u03b3=0':  mf_d,
        f'MF {n_steps}s \u03b3=1':  mf_s,
    }, title=f'{n_steps}-Step Comparison')

## 12. $\gamma$-Sweep (CTM & MF only, 5 steps)

CM excluded (locked to $\gamma = 1$).

In [ ]:
gammas = [0.0, 0.25, 0.5, 0.75, 1.0]
ctm_sweep = {f'CTM \u03b3={g:.2f}': sample(ctm, 'ctm', N, steps=5, gamma=g).cpu().numpy() for g in gammas}
mf_sweep  = {f'MF \u03b3={g:.2f}': sample(mf, 'mf', N, steps=5, gamma=g).cpu().numpy() for g in gammas}
plot_samples(data_np, ctm_sweep, title='CTM: \u03b3-sweep (5 steps)')
plot_samples(data_np, mf_sweep,  title='MF: \u03b3-sweep (5 steps)')

## 13. Interactive Explorer

When CM is selected, the $\gamma$ slider is **disabled and locked to 1.0**.

*(Requires ipywidgets — works in Colab/Jupyter)*

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display

    model_map = {'CM': (cm, 'cm'), 'CTM': (ctm, 'ctm'), 'MF': (mf, 'mf')}

    def interactive_sample(model_name, steps, gamma):
        mdl, mtype = model_map[model_name]
        if mtype == 'cm': gamma = 1.0
        pts = sample(mdl, mtype, 5000, steps=steps, gamma=gamma).cpu().numpy()
        fig, ax = plt.subplots(figsize=(5.5, 4.5))
        ax.scatter(data_np[:5000, 0], data_np[:5000, 1], s=1, alpha=0.12, c='gray', label='data')
        if mtype == 'cm':
            lbl = f'CM | {steps} step(s) | γ=1 (fixed)'
        else:
            lbl = f'{model_name} | {steps} step(s) | γ={gamma:.2f}'
        ax.scatter(pts[:, 0], pts[:, 1], s=2, alpha=0.5, label=lbl)
        ax.set_xlim(-5, 7); ax.set_ylim(-4, 4); ax.set_aspect('equal')
        ax.legend(fontsize=9); ax.set_title(lbl)
        fig.tight_layout(); plt.show()

    w_model = widgets.Dropdown(options=['CM', 'CTM', 'MF'], value='CTM', description='Model:')
    w_steps = widgets.IntSlider(min=1, max=20, value=1, description='Steps:')
    w_gamma = widgets.FloatSlider(min=0.0, max=1.0, step=0.05, value=0.0, description='γ:')

    def on_model_change(change):
        is_cm = (change['new'] == 'CM')
        w_gamma.disabled = is_cm
        if is_cm: w_gamma.value = 1.0
    w_model.observe(on_model_change, names='value')

    out = widgets.interactive_output(interactive_sample,
              {'model_name': w_model, 'steps': w_steps, 'gamma': w_gamma})
    display(widgets.VBox([widgets.HBox([w_model, w_steps, w_gamma]), out]))

except ImportError:
    print('ipywidgets not available. Run in Colab/Jupyter for interactive sliders.')

## Summary

### CM (§11.2)

- **Network:**
  $$
  \mathbf{f}_\theta(\mathbf{x}_s, s)
  $$
- **Parameterisation:**
  $$
  \mathbf{f}_\theta(\mathbf{x}, s) = (1-s)\,\mathbf{x} + s\,F_\theta(\mathbf{x}, s)
  $$
- **Boundary:**
  $$
  \mathbf{f}(\mathbf{x}_0, 0) = \mathbf{x}_0
  $$
- **Main loss:** stop-gradient consistency.
- **Auxiliary loss:** none.
- **1-step generation:** supported.
- **$\gamma$-sampling:** $\gamma=1$ only for multi-step sampling.

### CTM (§11.4)

- **Network:**
  $$
  \mathbf{G}_\theta(\mathbf{x}_s, s, t)
  $$
- **Parameterisation:**
  $$
  \mathbf{G}_\theta(\mathbf{x}_s, s, t) = \mathbf{x}_s + (t-s)\,\mathbf{v}_\theta(\mathbf{x}_s, s, t)
  $$
- **Boundary:**
  $$
  \mathbf{G}(\mathbf{x}_s, s, s) = \mathbf{x}_s
  $$
- **Main loss:** stop-gradient semigroup loss.
- **Auxiliary loss:** diffusion/flow-matching loss through
  $$
  \mathbf{v}_\theta(\mathbf{x}_s, s, s)
  $$
- **1-step generation:** supported.
- **$\gamma$-sampling:** any $\gamma \in [0,1]$.

### MF (§11.5)

- **Network:**
  $$
  \mathbf{h}_\theta(\mathbf{x}_s, s, t)
  $$
- **Parameterisation:** raw MLP average-drift predictor.
- **Boundary:** not imposed directly.
- **Main loss:** MeanFlow identity with JVP.
- **Auxiliary/anchor mode:** flow matching through
  $$
  \mathbf{h}_\theta(\mathbf{x}_s, s, s)
  $$
  with $p_{\text{fm}} = 25\%$ in this notebook.
- **1-step generation:** supported.
- **$\gamma$-sampling:** any $\gamma \in [0,1]$.

### Why auxiliary losses matter

**CTM** (§11.4.3): Semigroup loss has weak gradients when $t \approx s$ ($\because (1-t/s) \to 0$). $\mathcal{L}_{\text{DM}}$ supervises $\mathbf{v}_\theta(\mathbf{x}_s, s, s) \approx \boldsymbol{\epsilon} - \mathbf{x}_0$ — **first-order slope matching**.

**MF** (Tab. 1a): $p_{\text{fm}} = 25\%$ anchors $\mathbf{h}_\theta(\cdot, s, s)$ to correct local velocity. $0\%$ $\Rightarrow$ 1-step fails. $100\%$ $\Rightarrow$ pure FM, no 1-step.

### Why CM cannot do $\gamma < 1$

CM only outputs $\hat{\mathbf{x}}_0 = \mathbf{f}_\theta(\mathbf{x}_s, s)$. To reach $\tau > 0$: $\mathbf{x}_\tau = (1-\tau)\hat{\mathbf{x}}_0 + \tau\boldsymbol{\epsilon}_{\text{new}}$ — inherently $\gamma = 1$. CTM/MF can directly compute $\mathbf{G}_\theta(\mathbf{x}_s, s, \tau)$ or $\mathbf{x}_s - (s-\tau)\mathbf{h}_\theta(\mathbf{x}_s, s, \tau)$.